# 02 · Perfil de calidad
Cuántos registros tienen cada problema, en el dataset completo.


In [1]:
from pathlib import Path
import json, glob
import pandas as pd

LANDING = Path('../data/datalake/landing')
CSV = ['customers_orgs', 'users', 'resources', 'support_tickets',
       'marketing_touches', 'nps_surveys', 'billing_monthly']
tablas = {n: pd.read_csv(LANDING / f'{n}.csv') for n in CSV}

# eventos: leemos línea por línea con json para ver el tipo ORIGINAL de cada valor
archivos = sorted(glob.glob(str(LANDING / 'usage_events_stream' / '*.jsonl')))
filas = []
for i, archivo in enumerate(archivos):
    with open(archivo) as f:
        for linea in f:
            evento = json.loads(linea)
            evento['archivo_nro'] = i          # en qué micro-lote llegó
            filas.append(evento)
ev = pd.DataFrame(filas)
ev['ts'] = pd.to_datetime(ev['timestamp'], utc=True)
print(f'{len(archivos)} archivos · {len(ev):,} eventos')

120 archivos · 43,200 eventos


In [2]:
#nulos y claves de cada tabla

CLAVE = {'customers_orgs': 'org_id', 'users': 'user_id', 'resources': 'resource_id',
         'support_tickets': 'ticket_id', 'marketing_touches': 'touch_id',
         'nps_surveys': None, 'billing_monthly': 'invoice_id'}

for n, df in tablas.items():
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0].to_dict()
    unica = df[CLAVE[n]].is_unique if CLAVE[n] else 'varias filas por org'
    print(f'{n:18} filas={len(df):5}  clave única={unica}  nulos={nulos}')

customers_orgs     filas=   80  clave única=True  nulos={'nps_score': 11}
users              filas=  800  clave única=True  nulos={'last_login': 139}
resources          filas=  400  clave única=True  nulos={'tags_json': 83}
support_tickets    filas= 1000  clave única=True  nulos={'resolved_at': 240, 'csat': 254}
marketing_touches  filas= 1500  clave única=True  nulos={}
nps_surveys        filas=   92  clave única=varias filas por org  nulos={'nps_score': 19, 'comment': 10}
billing_monthly    filas=  240  clave única=True  nulos={'credits': 137}


In [3]:
#los eventos (tipos, nulos y versiones)

print('Tipo de "value":', ev['value'].map(lambda x: type(x).__name__).value_counts().to_dict())
print('value nulo:', ev['value'].isna().sum(), '| unit nula:', ev['unit'].isna().sum())
print('event_id repetidos:', ev['event_id'].duplicated().sum())
print()
print(ev.groupby('schema_version')['ts'].agg(['count', 'min', 'max']))
print()
print('Relación metric → unit:')
print(ev.dropna(subset=['unit']).groupby('metric')['unit'].unique())

Tipo de "value": {'float': 41014, 'str': 1309, 'NoneType': 877}
value nulo: 877 | unit nula: 2075
event_id repetidos: 0

                count                       min                       max
schema_version                                                           
1               10800 2025-07-03 00:02:00+00:00 2025-07-17 23:56:00+00:00
2               32400 2025-07-18 00:01:00+00:00 2025-08-31 23:58:00+00:00

Relación metric → unit:
metric
cpu_hours              [hours]
requests               [count]
storage_gb_hours    [gb_hours]
Name: unit, dtype: object


In [4]:
#costo negativo y picos

costo = ev['cost_usd_increment']
p99 = costo.quantile(0.99)
negativos = costo < 0
picos = costo > 5 * p99

print(f'mediana={costo.median():.2f}  p99={p99:.2f}  máximo={costo.max():.2f}  mínimo={costo.min():.2f}')
print(f'negativos: {negativos.sum()}  ·  picos (>5×p99): {picos.sum()}')
print(f'costo total: {costo.sum():,.0f} USD  ·  costo en picos: {costo[picos].sum():,.0f} USD '
      f'({costo[picos].sum() / costo.sum():.1%})')

mediana=1.00  p99=16.72  máximo=317.43  mínimo=-154.46
negativos: 216  ·  picos (>5×p99): 49
costo total: 147,434 USD  ·  costo en picos: 9,321 USD (6.3%)


In [5]:
por_archivo = ev.groupby('archivo_nro')['ts'].agg(['min', 'max'])
por_archivo['dias_que_cubre'] = (por_archivo['max'] - por_archivo['min']).dt.days
por_archivo.head(10)

,min,max,dias_que_cubre
archivo_nro,,,
0,2025-07-03 00:08:00+00:00,2025-08-31 16:32:00+00:00,59
1,2025-07-03 02:38:00+00:00,2025-08-31 23:51:00+00:00,59
2,2025-07-03 14:16:00+00:00,2025-08-31 20:47:00+00:00,59
3,2025-07-03 00:48:00+00:00,2025-08-31 23:51:00+00:00,59
4,2025-07-03 10:43:00+00:00,2025-08-31 20:40:00+00:00,59
5,2025-07-03 01:42:00+00:00,2025-08-31 23:56:00+00:00,59
6,2025-07-03 07:13:00+00:00,2025-08-31 21:21:00+00:00,59
7,2025-07-03 01:43:00+00:00,2025-08-31 20:28:00+00:00,59
8,2025-07-03 04:36:00+00:00,2025-08-31 21:34:00+00:00,59


In [6]:
#integridad referencial (hay "huerfanos?)
orgs = set(tablas['customers_orgs']['org_id'])
recursos = set(tablas['resources']['resource_id'])

for n, df in tablas.items():
    if n != 'customers_orgs':
        print(f'{n:18} org_id sin organización: {(~df["org_id"].isin(orgs)).sum()}')
print(f'{"usage_events":18} org_id sin organización: {(~ev["org_id"].isin(orgs)).sum()}')
print(f'{"usage_events":18} resource_id sin recurso: {(~ev["resource_id"].isin(recursos)).sum()}')

users              org_id sin organización: 0
resources          org_id sin organización: 0
support_tickets    org_id sin organización: 0
marketing_touches  org_id sin organización: 0
nps_surveys        org_id sin organización: 0
billing_monthly    org_id sin organización: 0
usage_events       org_id sin organización: 0
usage_events       resource_id sin recurso: 0


In [7]:
# el resto de los chequeos de negocio
b, t, u = tablas['billing_monthly'], tablas['support_tickets'], tablas['users']
nps_org, nps_enc = tablas['customers_orgs']['nps_score'], tablas['nps_surveys']['nps_score']

calidad = pd.Series({
    'eventos: value como texto':          (ev['value'].map(type) == str).sum(),
    'eventos: value nulo':                ev['value'].isna().sum(),
    'eventos: unit nula':                 ev['unit'].isna().sum(),
    'eventos: costo negativo':            negativos.sum(),
    'eventos: pico de costo':             picos.sum(),
    'billing: subtotal negativo':         (b['subtotal'] < 0).sum(),
    'billing: USD con tipo de cambio ≠ 1': ((b['currency'] == 'USD') & (b['exchange_rate_to_usd'].round(4) != 1)).sum(),
    'orgs: NPS fuera de 0–10':            ((nps_org < 0) | (nps_org > 10)).sum(),
    'encuestas: NPS fuera de 0–10':       ((nps_enc < 0) | (nps_enc > 10)).sum(),
    'tickets: CSAT > 5':                  (t['csat'] > 5).sum(),
    'tickets: abiertos (sin resolved_at)': t['resolved_at'].isna().sum(),
    'users: last_login antes de created_at': (pd.to_datetime(u['last_login']) < pd.to_datetime(u['created_at'])).sum(),
}, name='registros')
calidad

eventos: value como texto                1309
eventos: value nulo                       877
eventos: unit nula                       2075
eventos: costo negativo                   216
eventos: pico de costo                     49
billing: subtotal negativo                 13
billing: USD con tipo de cambio ≠ 1       160
orgs: NPS fuera de 0–10                    59
encuestas: NPS fuera de 0–10               66
tickets: CSAT > 5                          29
tickets: abiertos (sin resolved_at)       240
users: last_login antes de created_at     232
Name: registros, dtype: int64

In [8]:
#guarda la evidencia
EVIDENCIA = Path('../evidence'); EVIDENCIA.mkdir(exist_ok=True)
calidad.to_csv(EVIDENCIA / 'perfil_calidad.csv')
print('Guardado en', (EVIDENCIA / 'perfil_calidad.csv').resolve())

Guardado en /home/sebastian/Documentos/GitHub/facultad-main/mineria de datos II - miercoles/evidence/perfil_calidad.csv


## Conclusiones del perfil

### 1. Orden de llegada de los eventos
Cada uno de los 120 archivos cubre **59 días** (del 03/07 al 31/08/2025). Si los eventos llegaran en orden, cada archivo cubriría minutos u horas. Por lo tanto, **cada micro-lote trae eventos mezclados de todo el período: la llegada está completamente desordenada.**

Consecuencias para el diseño:
- Se particiona por **fecha del evento** (`event_date`), no por fecha de llegada.
- Un **watermark corto** de Spark descartaría la mayoría de los eventos por "tardíos". Hay que diseñar el streaming para tolerar eventos atrasados.

### 2. Integridad referencial
**No hay huérfanos:** todos los `org_id` existen en `customers_orgs` y todos los `resource_id` de los eventos existen en `resources`.

Es una **buena noticia**: al cruzar eventos con organizaciones y recursos en Silver (joins) no se pierde ninguna fila. Igual el chequeo queda como regla del pipeline, porque datos futuros podrían traer huérfanos.

### 3. Problema más grave para FinOps
La gravedad se mide por el **impacto en la decisión**, no por la cantidad de filas:

1. **Picos de costo:** son solo 49 eventos (0,1 %) pero concentran el **6,3 % del costo** (~9.300 USD de 147.434). Si son errores, inflan la factura; si son reales, son justo lo que FinOps necesita detectar. Por eso se **marcan con un flag, no se borran**.
2. **Moneda inconsistente en billing:** las 160 facturas en USD tienen tipo de cambio ≠ 1 y las facturas en ARS tienen subtotales en magnitud de USD. Afecta **todo el revenue**, no unas pocas filas.
3. **Menos grave:** los 1.309 `value` como texto. Se corrigen con una conversión y no se pierde información.